[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gravitational_waves/waves.ipynb)

# Gravitational Waves of a Closed Universe

This example computes the linear gravitational waves of a closed universe, the round three-sphere triangulated by regular tetrahedra of the 600-cell, and observes them from inside through small spring-bound crystals. Regge calculus represents curved space through the edge lengths of a tetrahedral triangulation, concentrating curvature at the edges as the deficit by which the dihedral angles around an edge fall short of a full turn. Treating each tetrahedron as a spherical simplex of the round three-sphere places the unperturbed 600-cell at rest with zero deficit on every edge, whereas flat tetrahedra leave a background deficit on every edge even without waves. Perturbing the squared chord lengths creates deficit angles that restore them against relativity's kinetic inertia, and the slowest mode squashes the sphere along one invariant direction while stretching it along an orthogonal direction.

Points on the unit three-sphere are modelled as antivectors, and the term ***simplex gradient*** here means the plane through a corner's opposite facet scaled to pair to one at that corner, so `gradients & point` reads the corner weights across a tetrahedron. Symmetric products of the simplex gradients weighted by the squared chord lengths form the cone metric from the sphere's centre as a ***duality map*** from points to planes, and pairing the simplex gradients through its inverse gives the cosines of the spherical dihedral angles. Pulling the kinetic mass map back across the vertex-motion operator as `moves.adjugate()(masses(moves))` projects out vertex motions with one linear solve, while smooth left-isoclinic frame motors carry each tetrahedron's metric perturbation back to a shared tangent space at the origin where vertex strains `strain(step) & step` blend smoothly across space and time. Unperturbed ray tracing across tetrahedral charts appears in [rendering.ipynb](rendering.ipynb), the view projected onto the two-sphere below the Hopf circles appears in [hopf_projection.ipynb](hopf_projection.ipynb), and `examples.relativity.gravitational_waves.web.page()` builds an interactive WebGPU flight through the ringing 600-cell.

In the notation of Regge calculus, as laid out by [Bahr and Dittrich (2009)](https://arxiv.org/abs/0907.4323), the action of spherical tetrahedra is $S = \sum_e \ell_e\,\epsilon_e + 2V$ over edge arc lengths $\ell_e$, deficit angles $\epsilon_e = 2\pi - \sum_{\sigma \supset e}\theta_{\sigma,e}$ computed from spherical cosine laws or Cayley–Menger determinants, and volume $V$, whose angle variations cancel by Schläfli's identity $\sum_e \ell_e\,\delta\theta_e = 2\,\delta V$ to leave $\delta S = \sum_e \epsilon_e\,\delta\ell_e$; linearizing in the squared chord perturbations $q$ yields $M\ddot q = -K q$ with the discrete DeWitt supermetric $M_{ee'} = \tfrac{1}{4}\sum_\sigma V_\sigma(g^{ik}g^{jl} + g^{il}g^{jk} - 2g^{ij}g^{kl})$ projected orthogonal to discrete diffeomorphisms and conformal rescalings via $(D^{\mathsf T}MD)\,u = D^{\mathsf T}M\,q$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from numga.topology import Topology

from examples.util.animation import show_states
from examples.relativity.gravitational_waves import numerical, render, scenarios

ga = Algebra("x+y+z+w+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
# Volume-integrated quantities: the Regge action and momenta conjugate to squared edge lengths.
Amount = ga.gatype.antiscalar()                                                 # Antiscalar-
# Simplex gradients across tetrahedra and polar planes through the origin.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within(Algebra("x+y+z+"))                                # Vector+
# Points of the three-sphere, tangent steps at the origin, frame motors, and bivector rates.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Through.derive.dual                                                      # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Rate = ga.gatype.bivector()                                                     # Bivector+
# Strain, frame-turning, frame-rolling, and step-stretching maps on steps at the origin.
Strain = ga.gatype((Through, Step))                                             # Plane+ <- Point-
Turning = ga.gatype((Rate, Step))                                               # Rate+ <- Point-
Rolling = ga.gatype((Rate, Step))                                               # Rate+ <- Point-
Stretching = ga.gatype((Step, Step))                                            # Point- <- Point-
Translating = ga.gatype((Step, Through))                                        # Point- <- Plane+
Moving = ga.gatype((Rate.derive.touching(Algebra("w+")), Step))                 # Rate+ <- Point-

# Equatorial plane `w` and the origin, its pole on the three-sphere.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
# Step of the origin across a plane through it; its inverse sends each step to its polar plane as a duality map.
translating = origin.commutator(equator ^ Through).cast(Translating)            # [] Point- <- Plane+
polar = translating.inverse()                                                   # [] Plane+ <- Point-
moving = equator ^ polar(Step)                                                  # [] Rate+ <- Point-
settling = 1e-10

## The frame at each place

In four dimensions, multiplying a bivector rate by the extrinsic pseudoscalar produces the rate in the orthogonal plane, and because the pseudoscalar squares to one it splits every rate into an invariant half and a flipped half. Each half turns by equal angles in two orthogonal planes, moving every point of the three-sphere the same distance along a great circle. The quotient `place / origin` turns the origin to `place` in their great-circle plane; keeping its pseudoscalar-invariant half with unit scalar part in the other half yields a smooth frame motor `frame(place)` everywhere on the sphere. Along any tangent step, the frames roll by `moving(step) + turning(step)`, while a free ray's heading stays constant in the frames along its great circle.

In [ ]:
# Extrinsic pseudoscalar mapping each rate to the rate in the orthogonal plane, and the orthogonal frame-turning map.
pseudoscalar = mv.xyzw                                                          # [] Pseudoscalar+
turning = moving * pseudoscalar                                                 # [] Rate+ <- Point-


def frame(places: Point) -> Motor:
    """Smooth left-isoclinic frame motors `Motor+` carry the origin to `places` by turning equally in the great-circle
    plane and its orthogonal complement."""
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The 600-cell at rest

The 600-cell places 120 vertices on the unit three-sphere, and its rest squared edge lengths are the squared chords between adjacent vertices. Read as a spherical simplex, each tetrahedron spans a cone from the sphere's centre whose metric is fixed by the simplex gradients and the squared chord lengths; pairing the opposite simplex gradients through the inverse cone metric gives the cosines of the dihedral angles around every edge. By Schläfli's identity for spherical tetrahedra, the dihedral-angle derivatives cancel against twice the volume in the action, leaving each edge's deficit angle to drive its squared chord length.

In [ ]:
cell_topology, cell_vertices = scenarios.six_hundred_cell()
V, E, F, T = cell_topology.n_elements
chords = cell_topology.d[0] * cell_vertices                                     # Point-[E]
rest = chords.scalar_norm_squared()                                             # Scalar+[E]
corners = cell_topology.incidence[3, 0] * cell_vertices                         # [4] Point-[T]
# Simplex gradients pair to one at their own corner and vanish on the opposite facet.
facing = cell_topology.off[3, 0] * cell_vertices                                # [3, 4] Point-[T]
facets = facing[0] & facing[1] & facing[2]                                      # [4] Plane-[T]
gradients = facets / (facets & corners)                                         # [4] Plane+[T]
# Chart hyperplane through all four corners, and symmetric products of simplex gradients forming the cone metric.
radial = gradients.sum(axis=0)                                                  # Plane+[T]
tails, heads = cell_topology.corners[3, 1]
cone_basis = -(gradients[tails] * (gradients[heads] & Point) + gradients[heads] * (gradients[tails] & Point)) / 2   # [6] (Plane+ <- Point-)[T]
cone_map = (spdiag(cone_basis) * cell_topology.incidence[3, 1]).sum(axis=0)     # (Plane+ <- Point-)[T <- E]
off_tails, off_heads = cell_topology.complements[3, 1]


def deficits(squared_lengths: Scalar) -> Scalar:
    """Spherical dihedral angles around each edge determine the deficit angles `Scalar+[E]` from `squared_lengths`."""
    cone = radial * (radial & Point) + cone_map * squared_lengths               # (Plane+ <- Point-)[T]
    pairings = gradients[:, None] & cone.batch().inverse().field()(gradients[None, :])   # [4, 4] Scalar+[T]
    off_edges = pairings[off_tails, off_heads]                                  # [6] Scalar+[T]
    sides = pairings[off_tails, off_tails] * pairings[off_heads, off_heads]     # [6] Scalar+[T]
    angles = (-off_edges / sides.square_root()).arccos()                        # [6] Scalar+[T]
    return 2 * np.pi - (cell_topology.incidence[3, 1].adjugate() * angles).sum(axis=-1)


def slopes(squared_lengths: Scalar) -> Amount:
    """Edge deficit angles weighted by the arc-length derivative give the Regge action slopes `Amount-[E]`."""
    return (-deficits(squared_lengths) / (squared_lengths * (1 - squared_lengths / 4)).square_root()).dual()

## The inertia of space and the waves

Relativity's kinetic term on the flat tetrahedra maps squared-length rates to volume-integrated momenta `masses`. Because every deficit vanishes on the rest chords, the linearized stiffness depends purely on how the dihedral angles vary through the inverse cone metric, whose derivative with respect to an edge's squared chord length reduces to products of the simplex-gradient pairings. Vertex motions in four-space stretch or slide the triangulation without propagating waves; pulling `masses` back across the vertex-motion map as `moves.adjugate()(masses(moves))` removes the mass-nearest vertex motion at each step.

In [ ]:
# Centred simplex gradients, flat metrics, and volumes assemble relativity's kinetic mass map on the edges.
centred_gradients = gradients - radial / 4                                      # [4] Plane+[T]
flat_basis = -(centred_gradients[tails] * (centred_gradients[heads] & Point)
               + centred_gradients[heads] * (centred_gradients[tails] & Point)) / 2   # [6] (Plane+ <- Point-)[T]
flat_map = (spdiag(flat_basis) * cell_topology.incidence[3, 1]).sum(axis=0)     # (Plane+ <- Point-)[T <- E]
flat = radial * (radial & Point) + flat_map * rest                              # (Plane+ <- Point-)[T]
spans = corners[1:] - corners[0]                                                # [3] Point-[T]
joins = corners[0] & corners[1] & corners[2] & corners[3]                       # Amount-[T]
volumes = ((flat(spans[0]) ^ flat(spans[1]) ^ flat(spans[2]) ^ radial) & joins).abs().square_root() / 6   # Scalar+[T]
pairings = centred_gradients[:, None] & flat.batch().inverse().field()(centred_gradients[None, :])   # [4, 4] Scalar+[T]
tet_masses = volumes / 4 * (pairings[tails[:, None], tails[None, :]] * pairings[heads[:, None], heads[None, :]]
                            + pairings[tails[:, None], heads[None, :]] * pairings[heads[:, None], tails[None, :]])   # [6, 6] Scalar+[T]
edges = cell_topology.incidence[3, 1]                                           # [6] Scalar+[T <- E]
masses = (edges[:, None].adjugate() * spdiag(tet_masses * Scalar.dual()) * edges[None, :]).sum(axis=0).sum(axis=0)   # (Amount- <- Scalar+)[E <- E]

# Variations of the spherical dihedral angles through the inverse cone metric assemble the Regge stiffness at rest.
cone = radial * (radial & Point) + cone_map * rest                              # (Plane+ <- Point-)[T]
cone_pairings = gradients[:, None] & cone.batch().inverse().field()(gradients[None, :])   # [4, 4] Scalar+[T]
first_off, second_off = off_tails[:, None], off_heads[:, None]
tail_ends, head_ends = tails[None, :], heads[None, :]
cross_pairings = (cone_pairings[first_off, tail_ends] * cone_pairings[second_off, head_ends]
                  + cone_pairings[first_off, head_ends] * cone_pairings[second_off, tail_ends]) / 2   # [6, 6] Scalar+[T]
self_pairings = (cone_pairings[first_off, tail_ends] * cone_pairings[first_off, head_ends] / cone_pairings[first_off, first_off]
                 + cone_pairings[second_off, tail_ends] * cone_pairings[second_off, head_ends] / cone_pairings[second_off, second_off])
edge_pairings = cone_pairings[first_off, second_off]                            # [6, 1] Scalar+[T]
sines = (cone_pairings[first_off, first_off] * cone_pairings[second_off, second_off] - edge_pairings.squared()).square_root()   # [6, 1] Scalar+[T]
angle_changes = (cross_pairings - edge_pairings * self_pairings / 2) / sines    # [6, 6] Scalar+[T]
roots = edges * (rest * (1 - rest / 4)).square_root()                           # [6] Scalar+[T]
stiffness = (edges[:, None].adjugate() * spdiag(angle_changes / roots[:, None] * Scalar.dual()) * edges[None, :]).sum(axis=0).sum(axis=0)   # (Amount- <- Scalar+)[E <- E]

In [ ]:
moves = spdiag(2 * chords.dot(Point)) * cell_topology.d[0]                      # (Scalar+ <- Point-)[E <- V]
# Regularizing with `settling` pins global rigid rotations of the sphere, which leave every edge length unchanged.
reach = moves.adjugate()(masses(moves))                                         # (Plane+ <- Point-)[V <- V]
reach = reach + settling * numerical.largest(reach) * Point.dual()


def wave_projection(changes: Scalar) -> Scalar:
    """Mass-orthogonal projection subtracts from `changes` `Scalar+[E]` the nearest vertex-motion perturbation."""
    return changes - moves(reach.solve(moves.adjugate()(masses(changes))))

## The squash

Pulling each edge's head into its tail's frame gives its tangent step at the origin, and the slowest quadrupole pattern lengthens edges along one invariant axis while shortening edges along an orthogonal axis uniformly in the frames. After projecting out vertex motions, the Rayleigh quotient of stiffness over mass gives its squared frequency on the 600-cell, within two percent of eight on edges spanning a fifth of a half turn.

In [ ]:
def round_steps(graph: Topology, places: Point) -> Point:
    """Pulling each edge's head back into its tail's frame gives the edge's tangent step `Point-[E]` at the origin."""
    tails, heads = graph.incidence[1, 0] * places                               # Point-[E], Point-[E]
    return (frame(tails).inverse() >> heads).cast(Step)


steps = round_steps(cell_topology, cell_vertices)                               # Point-[E]
squash = 2 * rest * ((mv.x & steps).squared() - (mv.y & steps).squared()) / (polar(steps) & steps)   # Scalar+[E]
pattern = wave_projection(squash)                                               # Scalar+[E]
squash_squared = (pattern & stiffness(pattern)).sites.sum() / (pattern & masses(pattern)).sites.sum()   # [] Scalar+
print(f"the squash's frequency squared: {float(squash_squared.to_array()):.3f}")

## Stepping through time

Projected leapfrog steps evolve the squared-length perturbations and their rates with one mass solve per step. Starting several periods before the rendered oscillation lets the spring-bound crystals settle into steady periodic motion and covers the light travel time across the closed universe.

In [ ]:
wave_swing = 0.12
n_steps_per_swing = 40
n_settling_swings = 8

In [ ]:
def waves_stepped(strains: Scalar, rates: Scalar, step: float) -> Iterator[tuple[Scalar, Scalar]]:
    """Yields squared edge-length perturbations and synchronized rates `(Scalar+[E], Scalar+[E])` across projected
    leapfrog steps."""
    restoring = wave_projection(masses.solve(stiffness(strains)))               # Scalar+[E]
    prior_rates = rates + (step / 2) * restoring                                # Scalar+[E]
    while True:
        next_rates = prior_rates - step * restoring                             # Scalar+[E]
        yield strains, (prior_rates + next_rates) / 2
        strains, prior_rates = strains + step * next_rates, next_rates
        restoring = wave_projection(masses.solve(stiffness(strains)))


frequency = np.sqrt(numerical.scalar_value(squash_squared))
period = 2 * np.pi / frequency
level_step = period / n_steps_per_swing
# History starts before the settling swings by half a great-circle turn so backward rays stay inside stored levels.
first_time = -(n_settling_swings + 1) * period - np.pi
n_levels = int(np.ceil((period - first_time) / level_step)) + 2
strains = wave_swing * np.cos(frequency * first_time) * pattern                 # Scalar+[E]
rates = -wave_swing * frequency * np.sin(frequency * first_time) * pattern      # Scalar+[E]
levels = [state for _, state in zip(range(n_levels), waves_stepped(strains, rates, level_step))]

## A smooth field of strain

Pulling each tetrahedron's cone-metric perturbation into the frame at its midpoint expresses every tetrahedron's strain on the same tangent steps at the origin, allowing vertex averaging and smooth corner-weight interpolation across tetrahedra and time levels. Spatial variation of the vertex strain across a tetrahedron contributes half its curl to the frame-rolling connection, while half the strain rate squeezes velocities as spatial slices stretch. Evaluating the vertex strain tables across all stored time levels takes several seconds.

In [ ]:
tet_frames = frame(cell_topology.incidence[3, 0].mean(axis=0) * cell_vertices)   # Motor+[T]
counts = (cell_topology.incidence[3, 0].adjugate() * mv.scalar(np.ones((T, 1))).field()).sum(axis=0)   # Scalar+[V]


def strain_field(changes: Scalar) -> tuple[Strain, Turning]:
    """Pulling each tetrahedron's cone-metric perturbation into its midpoint frame and averaging to vertices gives the
    vertex strain `(Plane+ <- Point-)[V]` and the tetrahedral curl turn `(Rate+ <- Point-)[T]`."""
    frame_steps = tet_frames >> Step                                            # (Point- <- Point-)[T]
    strains = (tet_frames.inverse() >> (cone_map * changes)(frame_steps)).cast(Strain)   # (Plane+ <- Point-)[T]
    at_vertices = (cell_topology.incidence[3, 0].adjugate() * strains).sum(axis=0) / counts   # (Plane+ <- Point-)[V]
    # Wedging pulled-back simplex gradients with corner strains forms half the spatial curl of the strain.
    corner_planes = (tet_frames.inverse() >> gradients).cast(Through)           # [4] Plane+[T]
    corner_strains = cell_topology.incidence[3, 0] * at_vertices                # [4] (Plane+ <- Point-)[T]
    return at_vertices, (corner_planes ^ corner_strains).sum(axis=0) / 2


level_fields = [strain_field(changes) for changes, _ in levels]
level_strains = stack([strain.batch() for strain, _ in level_fields])           # [n_levels, V] Plane+ <- Point-
level_turns = stack([turn.batch() for _, turn in level_fields])                 # [n_levels, T] Rate+ <- Point-
level_strain_rates = stack([strain_field(rates)[0].batch() for _, rates in levels])   # [n_levels, V] Plane+ <- Point-
level_traces = translating(level_strains).trace()                               # [n_levels, V] Scalar+

In [ ]:
def connection(tetrahedra: np.ndarray, places: Point, time: float) -> tuple[Rolling, Stretching, Stretching]:
    """Simplex-gradient corner weights and linear time weights assemble the frame-rolling connection
    `(Rate+ <- Point-)[V_sites]`, spatial stretch, and slice-rate squeeze `(Point- <- Point-)[V_sites]` at `places`."""
    level = (time - first_time) / level_step
    early_index, share = int(level), level - int(level)
    tet_reading = cell_topology.reading(tetrahedra, 3)                          # Scalar+[V_sites <- T]
    place_gradients = tet_reading * gradients                                   # [4] Plane+[V_sites]
    weights = (place_gradients & places) / (place_gradients.sum(axis=0) & places)   # [4] Scalar+[V_sites]

    def blended_field(values):
        """Linear time and corner interpolation evaluates vertex tables `[n_levels, V]` at `places`."""
        time_slice = (values[early_index] * (1 - share) + values[early_index + 1] * share).field()   # [V]
        return (weights * (tet_reading * (cell_topology.incidence[3, 0] * time_slice))).sum(axis=0)

    turn = tet_reading * (level_turns[early_index] * (1 - share) + level_turns[early_index + 1] * share).field()   # (Rate+ <- Point-)[V_sites]
    stretch = translating(blended_field(level_strains)(Step))                   # (Point- <- Point-)[V_sites]
    connection_rate = moving(Step + stretch / 2) + turning(Step - 2 * stretch + Step * (blended_field(level_traces) / 2)) + turn(Step)
    return connection_rate, stretch, translating(blended_field(level_strain_rates)) / 2

## Crystals in the waves

From inside a closed universe, bent light sways distant struts while free particles at rest keep fixed sphere coordinates as proper separations between them oscillate. Spring-bound atoms respond directly to those proper separations: as the wave stretches their bonds relative to their rest lengths, spring forces accelerate the atoms relative to free-falling neighbours, making small tetrahedral crystals genuine local strain gauges.

Three tetrahedral crystals sit side by side ahead of the eye with spring stiffnesses set to one sixteenth, one, and sixteen times the resonant value that puts the crystal's triply degenerate mode, each bond stretching as its opposite bond shortens, at the wave frequency, each damped at a quarter of critical. Symplectic Euler steps each atom's velocity in its local frame under the wave connection, slice stretch, and proper-separation spring forces: the soft crystal stretches with space, the stiff crystal holds its proper bond lengths, and the resonant crystal rings with amplified strain.

In [ ]:
tunings = np.array([1 / 16, 1, 16])                                             # [3]
critical_share = 0.25
crystal_step = 0.01
n_frames = 75

In [ ]:
pick = numerical.picking(cell_topology, cell_vertices)


def massive(bonds: Topology, rest: Scalar, spring_stiffness: Scalar, spring_damping: Scalar, starts: Point,
            start: float, step: float, times: np.ndarray) -> Iterator[tuple[Point, Scalar]]:
    """Yields atom positions `Point-[V_atoms]` and bond proper strains `Scalar+[E_bonds]` at `times` as unit-mass atoms
    step from rest at `starts` through the wave connection."""
    V_atoms, E_bonds, F_bonds, T_crystals = bonds.n_elements
    positions, lengths, time = starts.batch(), rest, start
    velocities = mv(Through, np.zeros((V_atoms, 3))).dual()                     # [V_atoms] Point-
    tails = bonds.incidence[1, 0][0]                                            # Scalar+[E_bonds <- V_atoms]
    for target in times:
        while time < target - step / 2:
            places = positions.field()                                          # Point-[V_atoms]
            connection_rate, stretch, squeeze = connection(pick(positions), places, time)   # (Rate+ <- Point-)[V_atoms], (Point- <- Point-)[V_atoms], (Point- <- Point-)[V_atoms]
            proper = origin.commutator((tails * connection_rate)(round_steps(bonds, places))).cast(Step)   # Point-[E_bonds]
            previous_lengths, lengths = lengths, proper.norm()                  # Scalar+[E_bonds], Scalar+[E_bonds]
            pulls = (spring_stiffness * (lengths - rest) + spring_damping * (lengths - previous_lengths) / step) / lengths   # Scalar+[E_bonds]
            forces = -(bonds.d[0].adjugate() * (pulls * proper)).batch()        # [V_atoms] Point-
            connection_rate, stretch, squeeze = connection_rate.batch(), stretch.batch(), squeeze.batch()   # [V_atoms] Rate+ <- Point-, Point- <- Point-, Point- <- Point-
            motion = velocities - stretch(velocities) / 2                       # [V_atoms] Point-
            carried_velocities = velocities - velocities.commutator(connection_rate(motion)).cast(Step) * step
            velocities = carried_velocities - squeeze(carried_velocities) * step + forces * step   # [V_atoms] Point-
            rolling = velocities - stretch(velocities) / 2                      # [V_atoms] Point-
            rolls = frame(positions) * (moving(rolling) * (-step / 2)).exp()    # [V_atoms] Motor+
            positions = rolls >> origin                                         # [V_atoms] Point-
            time += step
        yield positions.field(), lengths / rest - 1

In [ ]:
bonds, atoms = scenarios.crystals(scenarios.crystal_places())
V_atoms, E_bonds, F_bonds, T_crystals = bonds.n_elements
stiffnesses = frequency ** 2 / 2 * tunings                                      # [3]
dampings = critical_share * np.sqrt(2 * stiffnesses)                            # [3]
n_bonds_each = E_bonds // len(tunings)
bond_stiffness = mv.scalar(np.repeat(stiffnesses, n_bonds_each)[:, None]).field()   # Scalar+[E_bonds]
bond_damping = mv.scalar(np.repeat(dampings, n_bonds_each)[:, None]).field()        # Scalar+[E_bonds]
bond_steps = round_steps(bonds, atoms)                                          # Point-[E_bonds]
bond_rest = bond_steps.norm()                                                   # Scalar+[E_bonds]
eye_times = np.arange(n_frames) * period / n_frames                             # [n_frames]
kept_times = np.arange(eye_times[0] - np.pi, eye_times[-1] + level_step, level_step)
states = massive(bonds, bond_rest, bond_stiffness, bond_damping, atoms, kept_times[0] - n_settling_swings * period,
                 crystal_step, kept_times)

## The view from inside

Backward rays from the eye cross one tetrahedral chart per leg, subdividing each leg into midpoint geodesic rolls under the same interpolated wave connection that drives the crystals while evaluating both the waves and the crystal history at the retarded light time. Stepping the crystals and ray-tracing the animation takes about half a minute.

In [ ]:
leg_length = 0.16
n_rolls = 8
duration_ms = 40

In [ ]:
walk = '''
struct Tet {
    corners: vec4<u32>,
    neighbours: vec4<u32>,
    entries: vec4<u32>,
    gradients: array<Plane, 4>,
}

var<storage, read> tets: array<Tet>;
var<storage, read> strains: array<Strain>;
var<storage, read> strain_rates: array<Strain>;
var<storage, read> traces: array<f32>;
var<storage, read> turns: array<Turning>;
var<uniform> n_vertices: u32;
var<uniform> n_tets: u32;
var<uniform> first_time: f32;
var<uniform> level_step: f32;
var<uniform> origin: Point;
var<uniform> translating: Translating;
var<uniform> moving: Moving;
var<uniform> turning: Rolling;
var<uniform> leg_length: f32;
var<uniform> n_rolls: u32;

struct Leg {
    plane: Plane,
    chart: Point,
    segment: Point,
    length: f32,
    heading: Step,
    tet: u32,
    entry: u32,
}

struct Local {
    strain: Strain,
    strain_rate: Strain,
    trace: f32,
    turn: Turning,
}

fn local_waves(tet: u32, place: Point, time: f32) -> Local {
    let level = (time - first_time) / level_step;
    let early = u32(floor(level));
    let share = level - floor(level);
    let now = early * n_vertices;
    let later = now + n_vertices;
    let unit = tets[tet].gradients[0] + tets[tet].gradients[1] + tets[tet].gradients[2] + tets[tet].gradients[3];
    var waves: Local;
    for (var corner = 0u; corner < 4u; corner++) {
        let vertex = tets[tet].corners[corner];
        let weight = (tets[tet].gradients[corner] & place) / (unit & place);
        let early_weight = weight * (1.0 - share);
        let late_weight = weight * share;
        waves.strain = waves.strain + strains[now + vertex] * early_weight + strains[later + vertex] * late_weight;
        waves.strain_rate = waves.strain_rate + strain_rates[now + vertex] * early_weight
                            + strain_rates[later + vertex] * late_weight;
        waves.trace = waves.trace + traces[now + vertex] * early_weight + traces[later + vertex] * late_weight;
    }
    waves.turn = turns[early * n_tets + tet] * (1.0 - share) + turns[(early + 1u) * n_tets + tet] * share;
    return waves;
}

fn rolling(steps: Step, waves: Local) -> Rate {
    let stretch = translating(waves.strain(steps));
    return moving(steps + stretch * 0.5) + turning(steps - stretch * 2.0 + steps * (waves.trace * 0.5))
           + waves.turn(steps);
}

fn round_step(heading: Step, waves: Local) -> Step {
    return heading - translating(waves.strain(heading)) * 0.5;
}

fn heading_change(heading: Step, waves: Local) -> Step {
    return -heading.commutator(rolling(round_step(heading, waves), waves)).cast(Step)
           + translating(waves.strain_rate(heading)) * 0.5;
}

fn frame(place: Point) -> Motor {
    return ((1.0 + mv.xyzw) * (place / origin) + (1.0 - mv.xyzw)) * 0.5;
}

fn next_leg(place: Point, heading: Step, tet: u32, entry: u32, time: f32) -> Leg {
    let unit = tets[tet].gradients[0] + tets[tet].gradients[1] + tets[tet].gradients[2] + tets[tet].gradients[3];
    let chart = place / (unit & place);
    let start_waves = local_waves(tet, place, time);
    let step_end = (frame(place) * (moving(round_step(heading, start_waves)) * (-0.5 * leg_length)).exp()) >> origin;
    let step_segment = step_end / (unit & step_end) - chart;
    var share = 1.0;
    var exit = 4u;
    for (var corner = 0u; corner < 4u; corner++) {
        let change = tets[tet].gradients[corner] & step_segment;
        if (corner != entry && change < 0.0) {
            let run_out = max(-(tets[tet].gradients[corner] & chart) / change, 0.0);
            if (run_out < share) {
                share = run_out;
                exit = corner;
            }
        }
    }
    let length = acos(clamp(place.dot((chart + step_segment * share).normalized()), -1.0, 1.0));
    let n_leg_rolls = max(u32(ceil(f32(n_rolls) * share)), 1u);
    let roll = length / f32(n_leg_rolls);
    var leg_end = place;
    var end_heading = heading;
    for (var k = 0u; k < n_leg_rolls; k++) {
        let roll_time = time - roll * f32(k);
        let roll_waves = local_waves(tet, leg_end, roll_time);
        let half_heading = (end_heading + heading_change(end_heading, roll_waves) * (0.5 * roll)).normalized();
        let half_place = (frame(leg_end) * (moving(round_step(end_heading, roll_waves)) * (-0.25 * roll)).exp())
                         >> origin;
        let half_waves = local_waves(tet, half_place, roll_time - 0.5 * roll);
        leg_end = (frame(leg_end) * (moving(round_step(half_heading, half_waves)) * (-0.5 * roll)).exp()) >> origin;
        end_heading = (end_heading + heading_change(half_heading, half_waves) * roll).normalized();
    }
    var this_leg = Leg(unit, chart, leg_end / (unit & leg_end) - chart, length, end_heading, tet, 4u);
    if (exit < 4u) {
        this_leg.tet = tets[tet].neighbours[exit];
        this_leg.entry = tets[tet].entries[exit];
    }
    return this_leg;
}
'''

In [ ]:
walk_values = dict(gradients=gradients, strains=level_strains, strain_rates=level_strain_rates, traces=level_traces,
                   turns=level_turns, n_vertices=V, n_tets=T, first_time=first_time, level_step=level_step,
                   origin=origin, translating=translating, moving=moving, turning=turning, leg_length=leg_length,
                   n_rolls=n_rolls)
crystal_history = show_states(states, render.waves_view, walk, walk_values, cell_topology, cell_vertices,
                              scenarios.eye, eye_times, scenarios.guardrails(cell_vertices, scenarios.n_guardrail_sides),
                              bonds, kept_times, name="gravitational_waves_600_cell", duration_ms=duration_ms);

Twelve coloured guardrail decagons trace a Hopf family of great circles through the 600-cell while the remaining struts appear as faint grey ghosts. Bond colours encode proper strain from compressed blue to stretched red across the soft crystal on the left, the resonant crystal in the centre, and the stiff crystal on the right. Each atom of a crystal keeps its own colour, so that a crystal turning in the wave shows at a glance.

In [ ]:
# checks
first_strain, first_rate = levels[0]                                            # Scalar+[E], Scalar+[E]
last_strain, last_rate = levels[-1]                                             # Scalar+[E], Scalar+[E]
first_energy = ((first_strain & stiffness(first_strain)).sites.sum() + (first_rate & masses(first_rate)).sites.sum()) / 2
last_energy = ((last_strain & stiffness(last_strain)).sites.sum() + (last_rate & masses(last_rate)).sites.sum()) / 2
assert (deficits(rest).abs() < 1e-10).all()
assert (slopes(rest).antinorm() < 1e-10).all()
assert (7.5 < squash_squared) and (squash_squared < 8)
assert (last_energy - first_energy).abs() < 1e-2 * first_energy
peak_strains = stack([bond_strains.batch() for _, bond_strains in crystal_history]).abs().max(axis=0).reshape(3, -1).max(axis=1)
assert (peak_strains[1] > peak_strains[0]) and (peak_strains[1] > peak_strains[2])